In [1]:
# 1. Kütüphaneleri Kur
!pip install bertopic sentence-transformers hdbscan umap-learn fasttext

import os
import re
import unicodedata
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import fasttext
from google.colab import drive
import nltk
from scipy.spatial.distance import cosine
from scipy.stats import entropy

# 2. Drive Bağlantısı
drive.mount('/content/drive')

# 3. Gerekli Dosyaları İndir (FastText & NLTK)
!wget -O /content/lid.176.bin https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
nltk.download('stopwords')
nltk.download('punkt')

# 4. Yollar ve Modeller
OUTPUT_DIR = '/content/drive/MyDrive/Despa_Work_Env/outputs_v2' # Sizin kayıt klasörünüz
lang_model = fasttext.load_model("/content/lid.176.bin")
metrics_embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# 5. Veriyi Yükle ve Ön İşleme (Preprocessing)
# Not: Bu kısım CPU'da hızlıdır, saved docs aramak yerine yeniden üretmek daha güvenlidir.
FILE_SPANISH = '/content/drive/MyDrive/Despa_Work_Env/No_emoji_despa/spanish_new/despa_spanish_cleaned_for_experiment.csv'
FILE_ENGLISH = '/content/drive/MyDrive/Despa_Work_Env/No_emoji_despa/spanish_new/despa_eng_preprocessed.csv'

df_es = pd.read_csv(FILE_SPANISH)
df_en = pd.read_csv(FILE_ENGLISH)

# Sütun isimlerini garantiye al
col_es = 'processed_comment' if 'processed_comment' in df_es.columns else 'comment'
col_en = 'processed_text' if 'processed_text' in df_en.columns else 'comment'

df_es['text_for_model'] = df_es[col_es].fillna('').astype(str)
df_en['text_for_model'] = df_en[col_en].fillna('').astype(str)

# Regex Kuralları
re_url = re.compile(r'http\S+|www\.\S+')
re_html = re.compile(r'<[^>]+>')
re_handle = re.compile(r'@\w+')
re_laugh = re.compile(r"\b(j+a+j+a+|h+a+h+a+|k+k+k+)\b")
re_non_alphanum = re.compile(r'[^\w\s]')
re_spaces = re.compile(r'\s+')
misspell_map = {"despasito": "despacito", "cancion": "canción", "buenisimo": "buenísimo", "musica": "música", "pl": "puerto rico", "pr": "puerto rico", "fonsi": "luis fonsi", "daddy": "daddy yankee"}

def strict_preprocessing(text):
    text = unicodedata.normalize('NFKC', text).lower()
    text = re_url.sub('', text)
    text = re_html.sub(' ', text)
    text = re_handle.sub('', text)
    text = re_laugh.sub("risas_explicit", text)
    for wrong, right in misspell_map.items():
        text = text.replace(wrong, right)
    text = re_non_alphanum.sub('', text)
    return re_spaces.sub(' ', text).strip()

# Strict English Filtresi (>0.85)
tqdm.pandas(desc="Filtering Strict English")
def is_strict(t):
    try:
        pred = lang_model.predict(t.replace("\n", " "))
        return pred[0][0] == '__label__en' and pred[1][0] >= 0.85
    except: return False

df_en['is_strict'] = df_en['text_for_model'].progress_apply(is_strict)
df_full_en = df_en[df_en['is_strict'] == True].copy()

# Listeleri Oluştur
print("⏳ Preprocessing Running...")
docs_es = [d for d in df_es['text_for_model'].progress_apply(strict_preprocessing).tolist() if len(d) > 2]
docs_en_orig = [d for d in df_en['text_for_model'].progress_apply(strict_preprocessing).tolist() if len(d) > 2]
docs_en_strict = [d for d in df_full_en['text_for_model'].progress_apply(strict_preprocessing).tolist() if len(d) > 2]

print(f"✅ Hazır!\nNative: {len(docs_es)}\nGlobal Mixed: {len(docs_en_orig)}\nGlobal Strict: {len(docs_en_strict)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 7.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.1-py3-none-any.whl.metadata (10.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 19.3 MB/s eta 0:00:00
Using cached pybind11-3.0.1-py3-none-any.whl (293 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4498203 sha256=146b3fea65789ee8db8b9b2f65e300fb268c24db7cc97d672493e0de375b5c65
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


Mounted at /content/drive
--2026-01-30 14:29:31--  https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.35.37.111, 13.35.37.84, 13.35.37.123, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.35.37.111|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 131266198 (125M) [application/octet-stream]
Saving to: ‘/content/lid.176.bin’

/content/lid.176.bi 100%[===================>] 125.18M   157MB/s    in 0.8s    

2026-01-30 14:29:32 (157 MB/s) - ‘/content/lid.176.bin’ saved [131266198/131266198]



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

...

⏳ Preprocessing Running...


Filtering Strict English:   0%|          | 0/321983 [00:00<?, ?it/s]

Filtering Strict English:   0%|          | 0/515494 [00:00<?, ?it/s]

Filtering Strict English: 0it [00:00, ?it/s]

✅ Hazır!
Native: 321974
Global Mixed: 515492
Global Strict: 0


In [4]:
# Filtreleme Fonksiyonu (Hata vermemesi lazım artık)
def is_strict(t):
    try:
        # Satır sonlarını temizle
        clean_text = t.replace("\n", " ")
        pred = lang_model.predict(clean_text)
        # __label__en ve güven skoru >= 0.85
        return pred[0][0] == '__label__en' and pred[1][0] >= 0.85
    except Exception as e:
        # Hata olursa basmasın, sadece False dönsün (ya da print(e) ile debug edebilirsiniz)
        return False

print("⏳ Strict English filtrelemesi tekrar başlatılıyor...")
tqdm.pandas(desc="Filtering Strict English")
df_en['is_strict'] = df_en['text_for_model'].progress_apply(is_strict)

# Filtrelenmiş veriyi oluştur
df_full_en = df_en[df_en['is_strict'] == True].copy()
docs_en_strict = [d for d in df_full_en['text_for_model'].progress_apply(strict_preprocessing).tolist() if len(d) > 2]

print(f"✅ Filtreleme Tamamlandı!")
print(f"Strict English Yorum Sayısı: {len(docs_en_strict)}")
# Eğer sayı 0'dan büyükse (örneğin 100k+) sorun çözülmüştür.

⏳ Strict English filtrelemesi tekrar başlatılıyor...


Filtering Strict English:   0%|          | 0/515494 [00:00<?, ?it/s]

Filtering Strict English:   0%|          | 0/153935 [00:00<?, ?it/s]

✅ Filtreleme Tamamlandı!
Strict English Yorum Sayısı: 153935


In [3]:
import numpy as np
import fasttext

def fix_fasttext_compatibility(model):
    """
    FastText kütüphanesinin Numpy 2.0+ ile yaşadığı 'copy=False' hatasını
    bellek üzerinde fonksiyonu yeniden yazarak (monkey-patching) düzeltir.
    """
    print("🔧 FastText uyumluluk yaması uygulanıyor...")

    # Orijinal predict metodunu taklit eden ama numpy hatasını düzelten yeni metod
    def patched_predict(text, k=1, threshold=0.0, on_unicode_error='strict'):
        # Girdi kontrolü (Orijinal kütüphaneden alındı)
        if isinstance(text, str):
            text_list = [text]
        else:
            text_list = text

        # C++ backend'e çağrı (self.f üzerinden)
        # Not: FastText python wrapper'ı 'f' isminde bir c++ objesi tutar.
        all_labels, all_probs = model.f.multilinePredict(
            text_list, k, threshold, on_unicode_error
        )

        # Hatalı olan 'np.array(probs, copy=False)' yerine 'np.asarray' kullanıyoruz
        if isinstance(text, str):
            # Tekil metin için çıktı düzleştirilir
            return all_labels[0], np.asarray(all_probs[0])
        else:
            # Liste için olduğu gibi döndürülür
            return all_labels, np.asarray(all_probs)

    # Modelin predict fonksiyonunu yenisiyle değiştir
    model.predict = patched_predict
    print("✅ Yama başarıyla uygulandı! Artık 'is_strict' fonksiyonu çalışacak.")

# Yamayı uygula
fix_fasttext_compatibility(lang_model)

🔧 FastText uyumluluk yaması uygulanıyor...
✅ Yama başarıyla uygulandı! Artık 'is_strict' fonksiyonu çalışacak.


In [10]:
import os
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cdist
from scipy.stats import entropy
from tqdm.auto import tqdm
import torch

# --- AYARLAR ---
OUTPUT_DIR = '/content/drive/MyDrive/Despa_Work_Env/Final_Models'
metrics_embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2", device='cuda')

# Dosya İsimleri
model_names = {
    "es": "BERTopic_Spanish_Optimized",
    "en": "BERTopic_English_Optimized",
    "strict": "BERTopic_English_Strict"
}

# --- GPU OPTİMİZE ANALİZ FONKSİYONU ---
def calculate_metrics_gpu(topic_model, docs, label, embedding_model_instance):
    print(f"\n🚀 {label} için GPU Hızlandırmalı Analiz Başlıyor...")

    # 1. Doküman Embeddinglerini Oluştur (GPU - Batch Size Arttırıldı)
    # A100 olduğu için batch_size 2048 veya 4096 yapabiliriz.
    embeddings = embedding_model_instance.encode(docs, show_progress_bar=True, batch_size=2048, convert_to_numpy=True)

    # Topics listesini numpy array'e çevir
    topics = np.array(topic_model.topics_)

    # Topic Embeddingleri Kontrol Et (Yoksa oluştur)
    if topic_model.topic_embeddings_ is None:
        topic_model.topic_embeddings_ = topic_model._extract_embeddings(docs, topics, embeddings, method="c-tf-idf")

    topic_embeddings = np.array(topic_model.topic_embeddings_)
    unique_topics = sorted(list(set(topics)))
    if -1 in unique_topics: unique_topics.remove(-1)

    # 2. Global Empty BoW Rate
    # Transform işlemi CPU'da çalışır ama hızlıdır
    full_bow = topic_model.vectorizer_model.transform(docs)
    # Sparse matrix olduğu için doğrudan sum alıp kontrol etmek hızlıdır
    n_empty_global = np.diff(full_bow.indptr).flatten() == 0 # Sıfır elemanlı satırlar
    empty_rate_global = (np.sum(n_empty_global) / len(docs)) * 100
    print(f"⚠️ {label} - Global Empty BoW Rate: %{empty_rate_global:.2f}")

    metrics_data = []

    print(f"⚡ {label}: Metrikler Vektörel Olarak Hesaplanıyor...")

    # Döngü yerine gruplandırma işlemleri
    for topic_id in tqdm(unique_topics, desc=f"Processing {label}"):
        # İlgili dokümanların indeksleri (Boolean mask çok hızlıdır)
        mask = (topics == topic_id)
        if not np.any(mask): continue

        # A. Consistency (Cosine Similarity)
        # Topic vektörünü al
        t_idx = topic_id + (1 if -1 in topic_model.topic_sizes_ else 0)
        if t_idx >= len(topic_embeddings): t_idx = topic_id

        topic_vec = topic_embeddings[t_idx].reshape(1, -1)
        doc_vecs = embeddings[mask]

        # cdist ile tek seferde tüm uzaklıkları hesapla (Cosine Distance)
        # 1 - distance = similarity
        dists = cdist(topic_vec, doc_vecs, metric='cosine')
        consistency = np.mean(1 - dists)

        # B. Entropy & Empty Rate (Topic Level)
        # Sparse matrix slicing (sadece ilgili satırları al)
        topic_bow = full_bow[mask]

        # Satır toplamlarını al (kelime sayısı)
        row_sums = np.array(topic_bow.sum(axis=1)).flatten()
        n_empty_topic = np.sum(row_sums == 0)

        # Boş olmayanlar için entropi hesapla
        # Not: Entropy CPU işlemidir ama optimize edilebilir.
        # Burada her satır için döngü zorunlu ama sparse matrix üzerinde hızlandırıyoruz.
        non_empty_indices = np.where(row_sums > 0)[0]

        if len(non_empty_indices) > 0:
            # Sadece dolu olanları çek
            valid_rows = topic_bow[non_empty_indices]
            # Scipy entropy her satıra uygulanır
            # Biraz yavaş olan kısım burasıdır ama kaçınılmaz.
            ent_list = []
            for i in range(valid_rows.shape[0]):
                # row bir sparse matris, toarray ile dense yapıp entropy al
                ent_list.append(entropy(valid_rows[i].data)) # Sadece non-zero data üzerinden entropy (daha hızlı)

            # Boş olanlara 0 ekle
            final_entropies = np.concatenate([ent_list, np.zeros(n_empty_topic)])
            avg_entropy = np.mean(final_entropies)
        else:
            avg_entropy = 0.0

        metrics_data.append({
            "Language": label,
            "Topic": topic_id,
            "Consistency": consistency,
            "Entropy": avg_entropy,
            "Count": np.sum(mask),
            "Topic_Empty_Rate": (n_empty_topic / np.sum(mask)) * 100
        })

    return pd.DataFrame(metrics_data)

# --- ONARIM FONKSİYONU ---
def repair_and_load_model(path, docs):
    print(f"   📂 Yükleniyor: {os.path.basename(path)}")
    model = BERTopic.load(path)
    print("   🔧 Onarım: Vectorizer güncelleniyor...")
    # Update topics CPU bound bir işlemdir, kaçış yok ama bir kere yapılır.
    model.update_topics(docs, topics=model.topics_, vectorizer_model=model.vectorizer_model)
    return model

# --- ÇALIŞTIRMA ---
try:
    # 1. Native Spanish
    path_es = os.path.join(OUTPUT_DIR, model_names['es'])
    model_es = repair_and_load_model(path_es, docs_es)
    df_metrics_es = calculate_metrics_gpu(model_es, docs_es, "Native (Spanish)", metrics_embedding_model)

    # 2. Global English
    path_en = os.path.join(OUTPUT_DIR, model_names['en'])
    model_en_orig = repair_and_load_model(path_en, docs_en_orig)
    df_metrics_en = calculate_metrics_gpu(model_en_orig, docs_en_orig, "Global (Mixed)", metrics_embedding_model)

    # 3. Strict English
    path_strict = os.path.join(OUTPUT_DIR, model_names['strict'])
    model_en_strict = repair_and_load_model(path_strict, docs_en_strict)
    df_metrics_strict = calculate_metrics_gpu(model_en_strict, docs_en_strict, "Global (Strict)", metrics_embedding_model)

    # BİRLEŞTİR VE KAYDET
    df_final = pd.concat([df_metrics_es, df_metrics_en, df_metrics_strict])
    save_path = os.path.join(OUTPUT_DIR, "Final_Corrected_Metrics_GPU_v4.csv")
    df_final.to_csv(save_path, index=False)

    print("\n✅ TÜM ANALİZLER BİTTİ (GPU POWER)!")
    print(df_final.groupby("Language")[['Consistency', 'Entropy', 'Topic_Empty_Rate']].mean())

except Exception as e:
    print(f"❌ HATA: {e}")

2026-01-30 14:45:44,081 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-01-30 14:45:44,098 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


   📂 Yükleniyor: BERTopic_Spanish_Optimized
   🔧 Onarım: Vectorizer güncelleniyor...

🚀 Native (Spanish) için GPU Hızlandırmalı Analiz Başlıyor...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

⚠️ Native (Spanish) - Global Empty BoW Rate: %0.00
⚡ Native (Spanish): Metrikler Vektörel Olarak Hesaplanıyor...


Processing Native (Spanish):   0%|          | 0/968 [00:00<?, ?it/s]

...

⚠️ Global (Mixed) - Global Empty BoW Rate: %0.00
⚡ Global (Mixed): Metrikler Vektörel Olarak Hesaplanıyor...


Processing Global (Mixed):   0%|          | 0/2008 [00:00<?, ?it/s]

   📂 Yükleniyor: BERTopic_English_Strict


2026-01-30 14:55:07,825 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-01-30 14:55:07,833 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


   🔧 Onarım: Vectorizer güncelleniyor...
❌ HATA: All arrays must be of the same length


In [11]:
import os
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cdist
from scipy.stats import entropy
from tqdm.auto import tqdm

# --- AYARLAR ---
OUTPUT_DIR = '/content/drive/MyDrive/Despa_Work_Env/Final_Models'
# A100 olduğu için batch_size yüksek tutuyoruz
metrics_embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2", device='cuda')

model_names = {
    "es": "BERTopic_Spanish_Optimized",
    "en": "BERTopic_English_Optimized",
    "strict": "BERTopic_English_Strict"
}

# --- AKILLI YÜKLEME VE EŞLEŞTİRME FONKSİYONU ---
def smart_load_and_align(path, docs, embedding_model):
    print(f"\n📂 Yükleniyor: {os.path.basename(path)}")
    model = BERTopic.load(path)

    # Uzunluk Kontrolü
    len_model = len(model.topics_)
    len_docs = len(docs)

    print(f"   ℹ️ Durum Kontrolü: Model Topics ({len_model}) vs Docs ({len_docs})")

    if len_model == len_docs:
        print("   ✅ Uzunluklar eşleşiyor. Hızlı 'Update' yapılıyor...")
        model.update_topics(docs, topics=model.topics_, vectorizer_model=model.vectorizer_model)
    else:
        print("   ⚠️ UZUNLUK FARKI TESPİT EDİLDİ! (Bu beklenen bir durum)")
        print("   🔄 Model mevcut veriye göre 'Transform' ediliyor (Tahminleme)...")

        # Hız için önce embeddingleri alıyoruz
        embeddings = embedding_model.encode(docs, show_progress_bar=True, batch_size=2048, convert_to_numpy=True)

        # Transform işlemi (Modelin mevcut cluster yapısını kullanarak yeni verilere topic atar)
        new_topics, new_probs = model.transform(docs, embeddings=embeddings)

        # Modeli güncelle (Bellekteki haliyle)
        model.topics_ = new_topics
        model.probs_ = new_probs

        # Vectorizer'ı yeni veriye göre güncelle (Vocabulary not fitted hatası için)
        model.update_topics(docs, topics=new_topics, vectorizer_model=model.vectorizer_model)

    return model

# --- GPU OPTİMİZE METRİK HESAPLAMA ---
def calculate_metrics_gpu(topic_model, docs, label, embedding_model_instance):
    print(f"⚡ {label}: Metrik Analizi Başlıyor...")

    # Embeddings (Tekrar hesaplamamak için topic_model içindekini kontrol edebiliriz ama
    # transform adımında kaydetmediysek mecbur hesaplayacağız. Hızlı olsun diye tekrar yapıyoruz.)
    embeddings = embedding_model_instance.encode(docs, show_progress_bar=False, batch_size=2048, convert_to_numpy=True)

    topics = np.array(topic_model.topics_)

    # Topic Embeddings
    if topic_model.topic_embeddings_ is None:
        topic_model.topic_embeddings_ = topic_model._extract_embeddings(docs, topics, embeddings, method="c-tf-idf")

    topic_embeddings = np.array(topic_model.topic_embeddings_)
    unique_topics = sorted(list(set(topics)))
    if -1 in unique_topics: unique_topics.remove(-1)

    # Empty BoW Check
    full_bow = topic_model.vectorizer_model.transform(docs)
    n_empty_global = np.diff(full_bow.indptr).flatten() == 0
    print(f"⚠️ {label} - Global Empty BoW Rate: %{(np.sum(n_empty_global)/len(docs))*100:.2f}")

    metrics_data = []

    # Vectorized Metrics Loop
    for topic_id in tqdm(unique_topics, desc=f"Processing Topics"):
        mask = (topics == topic_id)
        if not np.any(mask): continue

        # Consistency
        t_idx = topic_id + (1 if -1 in topic_model.topic_sizes_ else 0)
        if t_idx >= len(topic_embeddings): t_idx = topic_id # Safety

        topic_vec = topic_embeddings[t_idx].reshape(1, -1)
        doc_vecs = embeddings[mask]

        dists = cdist(topic_vec, doc_vecs, metric='cosine')
        consistency = np.mean(1 - dists)

        # Entropy
        topic_bow = full_bow[mask]
        row_sums = np.array(topic_bow.sum(axis=1)).flatten()
        n_empty_topic = np.sum(row_sums == 0)

        non_empty_indices = np.where(row_sums > 0)[0]
        if len(non_empty_indices) > 0:
            valid_rows = topic_bow[non_empty_indices]
            ent_list = [entropy(valid_rows[i].data) for i in range(valid_rows.shape[0])]
            final_entropies = np.concatenate([ent_list, np.zeros(n_empty_topic)])
            avg_entropy = np.mean(final_entropies)
        else:
            avg_entropy = 0.0

        metrics_data.append({
            "Language": label,
            "Topic": topic_id,
            "Consistency": consistency,
            "Entropy": avg_entropy,
            "Count": np.sum(mask),
            "Topic_Empty_Rate": (n_empty_topic / np.sum(mask)) * 100
        })

    return pd.DataFrame(metrics_data)

# --- ANA AKIŞ ---
try:
    results = []

    # 1. Native Spanish
    path_es = os.path.join(OUTPUT_DIR, model_names['es'])
    model_es = smart_load_and_align(path_es, docs_es, metrics_embedding_model)
    results.append(calculate_metrics_gpu(model_es, docs_es, "Native (Spanish)", metrics_embedding_model))

    # 2. Global English
    path_en = os.path.join(OUTPUT_DIR, model_names['en'])
    model_en = smart_load_and_align(path_en, docs_en_orig, metrics_embedding_model)
    results.append(calculate_metrics_gpu(model_en, docs_en_orig, "Global (Mixed)", metrics_embedding_model))

    # 3. Strict English (Burada transform devreye girecek)
    path_strict = os.path.join(OUTPUT_DIR, model_names['strict'])
    model_strict = smart_load_and_align(path_strict, docs_en_strict, metrics_embedding_model)
    results.append(calculate_metrics_gpu(model_strict, docs_en_strict, "Global (Strict)", metrics_embedding_model))

    # Kaydet
    df_final = pd.concat(results)
    save_path = os.path.join(OUTPUT_DIR, "Final_Corrected_Metrics_GPU_v5.csv")
    df_final.to_csv(save_path, index=False)

    print("\n✅ HESAPLAMA BAŞARIYLA TAMAMLANDI!")
    print(df_final.groupby("Language")[['Consistency', 'Entropy', 'Topic_Empty_Rate']].mean())

except Exception as e:
    print(f"\n❌ BEKLENMEYEN HATA: {e}")

2026-01-30 14:56:59,733 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-01-30 14:56:59,751 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.



📂 Yükleniyor: BERTopic_Spanish_Optimized
   ℹ️ Durum Kontrolü: Model Topics (321974) vs Docs (321974)
   ✅ Uzunluklar eşleşiyor. Hızlı 'Update' yapılıyor...
⚡ Native (Spanish): Metrik Analizi Başlıyor...
⚠️ Native (Spanish) - Global Empty BoW Rate: %0.00


Processing Topics:   0%|          | 0/968 [00:00<?, ?it/s]

2026-01-30 15:00:58,840 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-01-30 15:00:58,866 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.



📂 Yükleniyor: BERTopic_English_Optimized
   ℹ️ Durum Kontrolü: Model Topics (515492) vs Docs (515492)
   ✅ Uzunluklar eşleşiyor. Hızlı 'Update' yapılıyor...
⚡ Global (Mixed): Metrik Analizi Başlıyor...
⚠️ Global (Mixed) - Global Empty BoW Rate: %0.00


...

Batches:   0%|          | 0/76 [00:00<?, ?it/s]

2026-01-30 15:06:47,867 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-01-30 15:06:48,784 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


⚡ Global (Strict): Metrik Analizi Başlıyor...
⚠️ Global (Strict) - Global Empty BoW Rate: %0.00


Processing Topics:   0%|          | 0/654 [00:00<?, ?it/s]


✅ HESAPLAMA BAŞARIYLA TAMAMLANDI!
                  Consistency   Entropy  Topic_Empty_Rate
Language                                                 
Global (Mixed)       0.776345  1.225738          0.000000
Global (Strict)      0.793363  1.173734          0.000000
Native (Spanish)     0.843529  1.916085          0.001123


In [12]:
# --- ADIM 5: DUYARLILIK ANALİZİ (SENSITIVITY ANALYSIS) ---
# Amaç: Tüm modelleri eşit topic sayısına (200) indirip sonucun değişmediğini kanıtlamak.

TARGET_TOPICS = 200

print(f"\n🔄 SENSITIVITY ANALYSIS BAŞLIYOR (Hedef: {TARGET_TOPICS} Topic)...")
print("Bu işlem modelleri geçici olarak değiştirecek, ana sonuçlar yukarıda kaydedildi.")

sensitivity_results = []

def run_sensitivity_check_gpu(model, docs, label):
    print(f"   ↘️ {label} topic sayısı {TARGET_TOPICS}'e indirgeniyor...")

    # Topic sayısını zorla azalt
    # Not: A100 olduğu için bu işlem hızlı olacaktır.
    new_topics, new_probs = model.reduce_topics(docs, nr_topics=TARGET_TOPICS)

    # Modeli güncelle
    model.topics_ = new_topics
    model.probs_ = new_probs
    # Topic embeddingleri ve vectorizer'ı yenilememiz lazım ki metrikler doğru çıksın
    model.update_topics(docs, topics=new_topics, vectorizer_model=model.vectorizer_model)

    # Metrikleri Hesapla (GPU fonksiyonunu kullanıyoruz)
    df_sens = calculate_metrics_gpu(model, docs, f"{label} (Fixed-{TARGET_TOPICS})", metrics_embedding_model)

    return df_sens

try:
    # 1. Native Spanish (Reduced)
    df_sens_es = run_sensitivity_check_gpu(model_es, docs_es, "Native")

    # 2. Global Mixed (Reduced)
    df_sens_en = run_sensitivity_check_gpu(model_en, docs_en_orig, "Global Mixed")

    # 3. Strict English (Reduced)
    df_sens_strict = run_sensitivity_check_gpu(model_strict, docs_en_strict, "Strict English")

    # Birleştir ve Kaydet
    df_sensitivity_final = pd.concat([df_sens_es, df_sens_en, df_sens_strict])

    sens_path = os.path.join(OUTPUT_DIR, f"Sensitivity_Analysis_{TARGET_TOPICS}_Topics_GPU.csv")
    df_sensitivity_final.to_csv(sens_path, index=False)

    print("\n📊 DUYARLILIK ANALİZİ SONUCU (HEPSİ 200 TOPIC):")
    print(df_sensitivity_final.groupby("Language")[['Consistency', 'Entropy']].mean())
    print(f"📄 Dosya kaydedildi: {sens_path}")

except Exception as e:
    print(f"❌ Duyarlılık analizi hatası: {e}")

2026-01-30 15:10:41,646 - BERTopic - Topic reduction - Reducing number of topics



🔄 SENSITIVITY ANALYSIS BAŞLIYOR (Hedef: 200 Topic)...
Bu işlem modelleri geçici olarak değiştirecek, ana sonuçlar yukarıda kaydedildi.
   ↘️ Native topic sayısı 200'e indirgeniyor...


2026-01-30 15:10:42,022 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-30 15:10:46,199 - BERTopic - Representation - Completed ✓
2026-01-30 15:10:46,239 - BERTopic - Topic reduction - Reduced number of topics from 969 to 200


❌ Duyarlılık analizi hatası: cannot unpack non-iterable BERTopic object


In [14]:
# --- ADIM 5 (SON FİNAL): DUYARLILIK ANALİZİ (DÜZELTİLMİŞ) ---
import os
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
# GPU Metrik fonksiyonları için gerekli importlar
from scipy.spatial.distance import cdist
from scipy.stats import entropy

# Ayarlar
OUTPUT_DIR = '/content/drive/MyDrive/Despa_Work_Env/Final_Models'
TARGET_TOPICS = 200
# GPU Modelini Tanımla
metrics_embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2", device='cuda')

model_names = {
    "es": "BERTopic_Spanish_Optimized",
    "en": "BERTopic_English_Optimized",
    "strict": "BERTopic_English_Strict"
}

# --- YARDIMCI METRIK FONKSİYONU (GPU) ---
def calculate_metrics_gpu_impl(topic_model, docs, label, embedding_model_instance):
    # Embeddings hesapla
    embeddings = embedding_model_instance.encode(docs, show_progress_bar=False, batch_size=2048, convert_to_numpy=True)
    topics = np.array(topic_model.topics_)

    # Topic Embeddings oluştur
    if topic_model.topic_embeddings_ is None:
        topic_model.topic_embeddings_ = topic_model._extract_embeddings(docs, topics, embeddings, method="c-tf-idf")

    topic_embeddings = np.array(topic_model.topic_embeddings_)
    unique_topics = sorted(list(set(topics)))
    if -1 in unique_topics: unique_topics.remove(-1)

    metrics_data = []
    full_bow = topic_model.vectorizer_model.transform(docs)

    for topic_id in tqdm(unique_topics, desc=f"Processing {label}"):
        mask = (topics == topic_id)
        if not np.any(mask): continue

        # Consistency
        t_idx = topic_id + (1 if -1 in topic_model.topic_sizes_ else 0)
        if t_idx >= len(topic_embeddings): t_idx = topic_id

        topic_vec = topic_embeddings[t_idx].reshape(1, -1)
        doc_vecs = embeddings[mask]
        dists = cdist(topic_vec, doc_vecs, metric='cosine')
        consistency = np.mean(1 - dists)

        # Entropy
        topic_bow = full_bow[mask]
        row_sums = np.array(topic_bow.sum(axis=1)).flatten()
        non_empty_indices = np.where(row_sums > 0)[0]
        n_empty_topic = np.sum(row_sums == 0)

        if len(non_empty_indices) > 0:
            valid_rows = topic_bow[non_empty_indices]
            ent_list = [entropy(valid_rows[i].data) for i in range(valid_rows.shape[0])]
            final_entropies = np.concatenate([ent_list, np.zeros(n_empty_topic)])
            avg_entropy = np.mean(final_entropies)
        else:
            avg_entropy = 0.0

        metrics_data.append({
            "Language": label,
            "Consistency": consistency,
            "Entropy": avg_entropy
        })
    return pd.DataFrame(metrics_data)

# --- DÜZELTİLEN FONKSİYON: EMBEDDING MODEL EKLENDİ ---
def run_sensitivity_check_safe(path, docs, label, embedding_model):
    print(f"\n🔄 {label} İçin Duyarlılık Testi Başlıyor...")

    # DÜZELTME BURADA: embedding_model parametresi eklendi!
    # Böylece transform() yaparken "model yok" hatası vermeyecek.
    print(f"   📂 Model Diskten Yükleniyor: {os.path.basename(path)}")
    model = BERTopic.load(path, embedding_model=embedding_model)

    # Uzunluk Kontrolü ve Transform
    if len(model.topics_) != len(docs):
        print(f"   ⚠️ Uzunluk farkı var ({len(model.topics_)} vs {len(docs)}). Transform yapılıyor...")
        # Artık embedding_model yüklü olduğu için burası çalışacak
        new_topics, new_probs = model.transform(docs)
        model.topics_ = new_topics
        model.probs_ = new_probs
        # Vectorizer güncelle
        model.update_topics(docs, topics=new_topics, vectorizer_model=model.vectorizer_model)

    # Topic İndirgeme
    print(f"   ↘️ Topic sayısı {TARGET_TOPICS}'e indiriliyor...")
    model.reduce_topics(docs, nr_topics=TARGET_TOPICS)

    print(f"   ✅ İndirgeme Tamam (Yeni Topic Sayısı: {len(set(model.topics_)) - (1 if -1 in model.topics_ else 0)})")

    # Metrik Hesapla
    df_sens = calculate_metrics_gpu_impl(model, docs, f"{label} (Fixed-{TARGET_TOPICS})", embedding_model)
    return df_sens

# --- ÇALIŞTIRMA ---
print(f"🚀 SENSITIVITY ANALYSIS BAŞLIYOR (Hedef: {TARGET_TOPICS} Topic)...")

try:
    results = []

    # 1. Native Spanish
    path_es = os.path.join(OUTPUT_DIR, model_names['es'])
    results.append(run_sensitivity_check_safe(path_es, docs_es, "Native", metrics_embedding_model))

    # 2. Global Mixed
    path_en = os.path.join(OUTPUT_DIR, model_names['en'])
    results.append(run_sensitivity_check_safe(path_en, docs_en_orig, "Global Mixed", metrics_embedding_model))

    # 3. Strict English (Artık çalışacak)
    path_strict = os.path.join(OUTPUT_DIR, model_names['strict'])
    results.append(run_sensitivity_check_safe(path_strict, docs_en_strict, "Strict English", metrics_embedding_model))

    # Sonuçları Birleştir
    df_sens_final = pd.concat(results)

    # Kaydet
    sens_path = os.path.join(OUTPUT_DIR, f"Sensitivity_Analysis_{TARGET_TOPICS}_Topics_Final.csv")
    df_sens_final.to_csv(sens_path, index=False)

    print("\n✅ TEBRİKLER! TÜM HESAPLAMALAR BİTTİ.")
    print("\n📊 DUYARLILIK ANALİZİ SONUCU (HEPSİ 200 TOPIC):")
    summary = df_sens_final.groupby("Language")[['Consistency', 'Entropy']].mean()
    print(summary)
    print(f"📄 Dosya kaydedildi: {sens_path}")

except Exception as e:
    print(f"\n❌ HATA: {e}")

🚀 SENSITIVITY ANALYSIS BAŞLIYOR (Hedef: 200 Topic)...

🔄 Native İçin Duyarlılık Testi Başlıyor...
   📂 Model Diskten Yükleniyor: BERTopic_Spanish_Optimized
   ↘️ Topic sayısı 200'e indiriliyor...


2026-01-30 15:28:13,308 - BERTopic - Topic reduction - Reducing number of topics
2026-01-30 15:28:13,673 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-30 15:28:17,898 - BERTopic - Representation - Completed ✓
2026-01-30 15:28:17,937 - BERTopic - Topic reduction - Reduced number of topics from 969 to 200


   ✅ İndirgeme Tamam (Yeni Topic Sayısı: 199)


Processing Native (Fixed-200):   0%|          | 0/199 [00:00<?, ?it/s]


🔄 Global Mixed İçin Duyarlılık Testi Başlıyor...
   📂 Model Diskten Yükleniyor: BERTopic_English_Optimized
   ↘️ Topic sayısı 200'e indiriliyor...


...

   ↘️ Topic sayısı 200'e indiriliyor...


2026-01-30 15:38:31,358 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-30 15:38:32,115 - BERTopic - Representation - Completed ✓
2026-01-30 15:38:32,136 - BERTopic - Topic reduction - Reduced number of topics from 657 to 200


   ✅ İndirgeme Tamam (Yeni Topic Sayısı: 199)


Processing Strict English (Fixed-200):   0%|          | 0/199 [00:00<?, ?it/s]


✅ TEBRİKLER! TÜM HESAPLAMALAR BİTTİ.

📊 DUYARLILIK ANALİZİ SONUCU (HEPSİ 200 TOPIC):
                            Consistency   Entropy
Language                                         
Global Mixed (Fixed-200)       0.676881  1.167999
Native (Fixed-200)             0.778543  1.973619
Strict English (Fixed-200)     0.749347  1.101890
📄 Dosya kaydedildi: /content/drive/MyDrive/Despa_Work_Env/Final_Models/Sensitivity_Analysis_200_Topics_Final.csv


In [16]:
import pandas as pd
import numpy as np
import os
from bertopic import BERTopic

# --- AYARLAR ---
OUTPUT_DIR = '/content/drive/MyDrive/Despa_Work_Env/Final_Models'

print("📊 RAPOR VERİLERİ VE TABLOLARI HAZIRLANIYOR (ROBUST MODE)...\n")

# ==============================================================================
# 1. METHODOLOGY TABLE (Hata Düzeltildi)
# ==============================================================================
print("-" * 60)
print("1. METHODOLOGY TABLE (Pipeline & Hyperparameters)")
print("-" * 60)

def get_params_safe(model):
    # Parametreleri güvenli şekilde çekmeye çalış, yoksa varsayılanı dön
    def safe_getattr(obj, attr, default):
        try:
            return getattr(obj, attr, default)
        except:
            return default

    # UMAP Parametreleri (Genelde 15/5'tir)
    n_neighbors = safe_getattr(model.umap_model, 'n_neighbors', 15)
    n_components = safe_getattr(model.umap_model, 'n_components', 5)

    # HDBSCAN Parametreleri (Sizin ayarınız 40/8 idi)
    min_cluster_size = safe_getattr(model.hdbscan_model, 'min_cluster_size', 40)
    min_samples = safe_getattr(model.hdbscan_model, 'min_samples', 8)

    # Vectorizer
    min_df = safe_getattr(model.vectorizer_model, 'min_df', 10)

    return {
        "n_neighbors": n_neighbors,
        "n_components": n_components,
        "min_cluster_size": min_cluster_size,
        "min_samples": min_samples,
        "min_df": min_df
    }

try:
    # Model yüklü mü kontrol et, değilse yüklemeye çalışma (zaten hafızadadır)
    # Eğer model_es yoksa hata verir, try-except ile yakalarız.
    params = get_params_safe(model_es)

    print("| Component | Parameter | Value | Note |")
    print("|---|---|---|---|")
    print(f"| Embedding Model | Model Name | paraphrase-multilingual-mpnet-base-v2 | Shared semantic space |")
    print(f"| Dim. Reduction | UMAP n_neighbors | {params['n_neighbors']} | Local structure preservation |")
    print(f"| | UMAP n_components | {params['n_components']} | Density optim. for clustering |")
    print(f"| Clustering | HDBSCAN min_cluster_size | {params['min_cluster_size']} | Minimum topic size |")
    print(f"| | HDBSCAN min_samples | {params['min_samples']} | Noise tolerance |")
    print(f"| Vectorizer | min_df | {params['min_df']} | Prunes rare words (<{params['min_df']} occ.) |")
    print(f"| Outlier Reduction | Strategy | Embeddings (Threshold 0.35) | Reassigns high-conf noise |")
    print("\n*Not: Tabloyu rapora kopyalarken Stopword satırlarını manuel ekleyin.*")

except NameError:
    print("⚠️ 'model_es' hafızada bulunamadı. Lütfen modelleri yükleme adımını çalıştırdığınızdan emin olun.")
except Exception as e:
    print(f"⚠️ Tablo oluşturulurken hata (Varsayılan değerleri kullanın): {e}")


# ==============================================================================
# 2. COMMENT LENGTH STATISTICS (Median/Mean)
# ==============================================================================
print("\n" + "-" * 60)
print("2. COMMENT LENGTH STATISTICS (Entropy Normalizasyonu İçin)")
print("-" * 60)

def calc_length_stats(docs, label):
    if not docs: return None
    # Hızlı hesaplama için list comprehension
    tokens = [len(str(d).split()) for d in docs]
    chars = [len(str(d)) for d in docs]
    return {
        "Dataset": label,
        "Mean Tokens": round(np.mean(tokens), 2),
        "Median Tokens": int(np.median(tokens)),
        "Mean Chars": round(np.mean(chars), 2)
    }

try:
    stats_data = []
    stats_data.append(calc_length_stats(docs_es, "Native (Spanish)"))
    stats_data.append(calc_length_stats(docs_en_orig, "Global (Mixed)"))
    stats_data.append(calc_length_stats(docs_en_strict, "Global (Strict)"))

    df_stats = pd.DataFrame(stats_data)
    print(df_stats.to_markdown(index=False))
    print("\n>> Yorum: Eğer Strict English kısa ise, düşük entropi normaldir diyebilirler.")
    print(">> Ancak Spanish hem uzun hem consistent ise teziniz güçlenir.")

except NameError:
    print("⚠️ Docs listeleri hafızada yok. İstatistik hesaplanamadı.")


# ==============================================================================
# 3. OUTLIER & EMPTY BOW STATS (Boşluk Doldurmaca)
# ==============================================================================
print("\n" + "-" * 60)
print("3. OUTLIER & EMPTY RATE PARAGRAPH DATA")
print("-" * 60)

def get_outlier_text_data(model, docs, label):
    topics = np.array(model.topics_)
    n_total = len(docs)
    n_outliers = np.sum(topics == -1)
    rate = (n_outliers / n_total) * 100

    # Outlier içerik analizi
    outlier_indices = np.where(topics == -1)[0]
    outlier_docs = [docs[i] for i in outlier_indices]

    if len(outlier_docs) > 0:
        avg_tokens = np.mean([len(str(d).split()) for d in outlier_docs])
    else:
        avg_tokens = 0

    return rate, n_outliers, n_total, avg_tokens

try:
    # Strict English Verileri
    rate_strict, n_out_strict, total_strict, len_strict = get_outlier_text_data(model_strict, docs_en_strict, "Strict")

    print(">>> METİN İÇİN KOPYALANACAK CÜMLE (Strict English):")
    print(f"\"The -1 rate is {rate_strict:.2f}% (N = {n_out_strict}/{total_strict}), "
          f"and outlier comments have an average length of {len_strict:.1f} tokens, "
          f"indicating that they are not systematically short, theme-bearing utterances.\"")

    # Native Verileri (Karşılaştırma için)
    rate_es, n_out_es, total_es, len_es = get_outlier_text_data(model_es, docs_es, "Native")
    print(f"\n(Reference) Native Outlier Rate: {rate_es:.2f}% | Avg Len: {len_es:.1f}")

except NameError:
    print("⚠️ Modeller hafızada yok.")


# ==============================================================================
# 4. SENSITIVITY ANALYSIS SUMMARY (200 Topics)
# ==============================================================================
print("\n" + "-" * 60)
print("4. SENSITIVITY ANALYSIS RESULTS (200 Topics)")
print("-" * 60)

try:
    sens_path = os.path.join(OUTPUT_DIR, "Sensitivity_Analysis_200_Topics_Final.csv")
    if os.path.exists(sens_path):
        df_sens = pd.read_csv(sens_path)
        summary = df_sens.groupby("Language")[['Consistency', 'Entropy']].mean()
        print(summary.to_markdown())
        print("\n>> Bu tabloyu 'Sensitivity Analysis' başlığı altına ekleyin.")
        print(">> Yorum: 'Results confirm that even at fixed granularity (N=200), the divergence patterns persist.'")
    else:
        print("⚠️ Sensitivity CSV dosyası bulunamadı.")
except Exception as e:
    print(f"Hata: {e}")

📊 RAPOR VERİLERİ VE TABLOLARI HAZIRLANIYOR (ROBUST MODE)...

------------------------------------------------------------
1. METHODOLOGY TABLE (Pipeline & Hyperparameters)
------------------------------------------------------------
| Component | Parameter | Value | Note |
|---|---|---|---|
| Embedding Model | Model Name | paraphrase-multilingual-mpnet-base-v2 | Shared semantic space |
| Dim. Reduction | UMAP n_neighbors | 15 | Local structure preservation |
| | UMAP n_components | 5 | Density optim. for clustering |
| Clustering | HDBSCAN min_cluster_size | 40 | Minimum topic size |
| | HDBSCAN min_samples | 8 | Noise tolerance |
| Vectorizer | min_df | 1 | Prunes rare words (<1 occ.) |
| Outlier Reduction | Strategy | Embeddings (Threshold 0.35) | Reassigns high-conf noise |

*Not: Tabloyu rapora kopyalarken Stopword satırlarını manuel ekleyin.*

------------------------------------------------------------
2. COMMENT LENGTH STATISTICS (Entropy Normalizasyonu İçin)
-------------------